In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11

C = {"navy": "#235796", "terra": "#CA6F6A", "midnight": "#20425B"}
fontsize = 18

K_fixed, T_fixed, L_fixed = 6, 50, 512

def style(ax, xlabel, title):
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=fontsize, color=C["midnight"])
    ax.set_facecolor("white")
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

fig = plt.figure(figsize=(18, 4), facecolor="white")
gs = gridspec.GridSpec(1, 3)
lines = []

# ── vs L ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0])
L = np.linspace(32, 512, 500)

tfm  = T_fixed * (L + K_fixed)**2
prop = T_fixed**2 + T_fixed * K_fixed * L

lines.append(ax.plot(L, tfm /1e6, color=C["terra"], lw=2.8,
        label="Decoder-Only\n$\\mathcal{O}(TL^2)$")[0])
lines.append(ax.plot(L, prop/1e6, color=C["navy"],  lw=2.8,
        label="Encoder-Decoder (w/ FS)                 \n$\\mathcal{O}(T^2+TKL)$")[0])
ax.fill_between(L, prop/1e6, tfm/1e6, alpha=0.10, color=C["terra"])

ax.set_ylabel("Relative compute (log scale)", fontsize=fontsize)
ax.set_xlim(32, 512)
style(ax, "Context length $L$", f"($K={K_fixed}$, $T={T_fixed}$)")

# ── vs K ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1])
K = np.arange(1, 16)

tfm2  = T_fixed * (L_fixed + K)**2
prop2 = T_fixed**2 + T_fixed * K * L_fixed

ax.plot(K, tfm2 /1e6, color=C["terra"], lw=2.8)
ax.plot(K, prop2/1e6, color=C["navy"],  lw=2.8)
ax.fill_between(K, prop2/1e6, tfm2/1e6, alpha=0.10, color=C["terra"])
ax.set_xlim(1, 15)
style(ax, "Rollout depth $K$",
      f"Parallelized Teacher Forcing  ($L={L_fixed}$, $T={T_fixed}$)")

# ── vs T ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2])
T = np.linspace(1, 1000, 1000)

tfm3  = T * (L_fixed + K_fixed)**2
prop3 = T**2 + T * K_fixed * L_fixed

ax.plot(T, tfm3 /1e6, color=C["terra"], lw=2.8)
ax.plot(T, prop3/1e6, color=C["navy"],  lw=2.8)
ax.fill_between(T, prop3/1e6, tfm3/1e6,
                where=tfm3 >= prop3, alpha=0.10, color=C["terra"])
ax.fill_between(T, prop3/1e6, tfm3/1e6,
                where=tfm3 < prop3,  alpha=0.10, color=C["navy"])
ax.set_xlim(1, 1000)
style(ax, "Forecast creation dates $T$", f"($L={L_fixed}$, $K={K_fixed}$)")

fig.legend(
    handles=lines,
    loc="center left",
    bbox_to_anchor=(0.92, 0.7),
    fontsize=fontsize - 2,
    frameon=True,
    borderaxespad=0.,
)

plt.savefig("./complexity_train.pdf",
            dpi=200, bbox_inches="tight", facecolor="white")


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11

C = {"navy": "#235796", "terra": "#CA6F6A", "midnight": "#20425B"}
fontsize = 18

K_fixed, T_fixed, L_fixed = 6, 50, 512

def style(ax, xlabel, title):
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=fontsize, color=C["midnight"])
    ax.set_facecolor("white")
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

fig = plt.figure(figsize=(18, 4), facecolor="white")
gs = gridspec.GridSpec(1, 3)
lines = []

# ── vs L ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0])
L = np.linspace(32, 512, 500)

tfm_nokv  = T_fixed * K_fixed * L**2
tfm_kv    = T_fixed * L**2 + T_fixed * K_fixed * L
prop_nokv = T_fixed**2 + T_fixed * K_fixed**2 * L
prop_kv   = T_fixed**2 + T_fixed * K_fixed * L

# shade between best-of-each-method (mirrors training plot story)
ax.fill_between(L, prop_kv/1e6, tfm_kv/1e6, alpha=0.12, color=C["terra"])

lines.append(ax.plot(L, tfm_nokv /1e6, color=C["terra"], lw=2.8,
        label="Decoder-Only\n$\\mathcal{O}(TKL^2)$")[0])
lines.append(ax.plot(L, tfm_kv   /1e6, color=C["terra"], lw=2.0, ls="--",
        label="Decoder-Only (w/ KV Cache)\n$\\mathcal{O}(TL^2+TKL)$")[0])
lines.append(ax.plot(L, prop_nokv/1e6, color=C["navy"],  lw=2.8,
        label="Encoder-Decoder (w/ FS)\n$\\mathcal{O}(T^2+TK^2L)$")[0])
lines.append(ax.plot(L, prop_kv  /1e6, color=C["navy"],  lw=2.0, ls="--",
        label="Encoder-Decoder (w/ FS, KV Cache)\n$\\mathcal{O}(T^2+TKL)$")[0])

ax.set_ylabel("Relative compute (log scale)", fontsize=fontsize)
ax.set_xlim(32, 512)
style(ax, "Context length $L$", f"($K={K_fixed}$, $T={T_fixed}$)")

# ── vs K ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1])
K = np.arange(1, 16)

tfm2_nokv  = T_fixed * K * L_fixed**2
tfm2_kv    = T_fixed * L_fixed**2 + T_fixed * K * L_fixed
prop2_nokv = T_fixed**2 + T_fixed * K**2 * L_fixed
prop2_kv   = T_fixed**2 + T_fixed * K * L_fixed

ax.fill_between(K, prop2_kv/1e6, tfm2_kv/1e6, alpha=0.12, color=C["terra"])

ax.plot(K, tfm2_nokv /1e6, color=C["terra"], lw=2.8)
ax.plot(K, tfm2_kv   /1e6, color=C["terra"], lw=2.0, ls="--")
ax.plot(K, prop2_nokv/1e6, color=C["navy"],  lw=2.8)
ax.plot(K, prop2_kv  /1e6, color=C["navy"],  lw=2.0, ls="--")
ax.set_xlim(1, 15)
style(ax, "Rollout depth $K$", f"($L={L_fixed}$, $T={T_fixed}$)")

# ── vs T ──────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[2])
T = np.linspace(1, 1000, 1000)

tfm3_nokv  = T * K_fixed * L_fixed**2
tfm3_kv    = T * L_fixed**2 + T * K_fixed * L_fixed
prop3_nokv = T**2 + T * K_fixed**2 * L_fixed
prop3_kv   = T**2 + T * K_fixed * L_fixed

ax.fill_between(T, prop3_kv/1e6, tfm3_kv/1e6,
                where=tfm3_kv >= prop3_kv, alpha=0.12, color=C["terra"])
ax.fill_between(T, prop3_kv/1e6, tfm3_kv/1e6,
                where=tfm3_kv < prop3_kv,  alpha=0.12, color=C["navy"])

ax.plot(T, tfm3_nokv /1e6, color=C["terra"], lw=2.8)
ax.plot(T, tfm3_kv   /1e6, color=C["terra"], lw=2.0, ls="--")
ax.plot(T, prop3_nokv/1e6, color=C["navy"],  lw=2.8)
ax.plot(T, prop3_kv  /1e6, color=C["navy"],  lw=2.0, ls="--")
ax.set_xlim(1, 1000)
style(ax, "Forecast creation dates $T$", f"($L={L_fixed}$, $K={K_fixed}$)")

# ── Shared legend ──────────────────────────────────────────────────────────
fig.legend(
    handles=lines,
    loc="center left",
    bbox_to_anchor=(0.92, 0.55),
    fontsize=fontsize - 2,
    frameon=True,
    borderaxespad=0.,
)

plt.savefig("./complexity_inference.pdf",
            dpi=200, bbox_inches="tight", facecolor="white")
